# Phase 36 -- a general query reparser: what it fixes and what it doesn't

Triggered by a direct question mid-investigation: after building two narrow, bespoke
fixes for q09 (a disambiguation note in `calc.py`) and q29 (a broadened coverage
rule), could a single general mechanism -- an LLM step that rewrites the question
into a more explicit version of itself before generation -- have solved these, and
could it also replace the OTHER bespoke fixes already in this codebase (the
contributions calculator, the sex-differentiated pension split)?

This notebook is a record of that investigation, not a from-scratch build -- the
underlying trials were run interactively and are reloaded here from the persisted
JSON files in `eval/`, at zero additional API cost.

**The mechanism**, `OpenAICompatibleGenerator._reparse_for_completeness` in
`src/generation.py`: a second LLM call that takes the question about to be shown to
the generator and rewrites it into a more explicit version of the SAME question --
if it asks who is entitled to something and there could be more than one eligible
category, the rewrite explicitly asks for every category and its own condition.
Appended AFTER the original question in the prompt (never replacing it), so the
person's own words still drive the answer's tone and the `calc.py` extractors
(`extract_sex`, `extract_salary`, etc.) still read the untouched original query.
**Adopted into production either way**, per direct instruction -- wired into
`generate()` unconditionally.


In [1]:
import json
from pathlib import Path

import pandas as pd

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "retrieval.py").exists()), None)
assert PROJECT, "run this notebook from inside the project tree"
EVAL_DIR = PROJECT / "eval"

def load(name):
    with open(EVAL_DIR / name, encoding="utf-8") as f:
        return json.load(f)


## 1. The prompt had to be tightened once before it was usable

First version asked the model to "make the question more explicit" with no further
guardrail. It worked (6/6 on q29) but invented NEW comparison axes the person never
asked about (e.g. added "does this differ by employer sector?" to a question that
never raised sector), roughly tripling answer length by pulling in tangential
articles that axis triggered. Tightened to forbid any new topic/angle and restrict
it to making the ORIGINAL question's own scope explicit -- e.g. "who is entitled"
becomes "every eligible category and that category's own condition," nothing more.


In [2]:
v1 = load("q29_reparser_test.json")
v2 = load("q29_reparser_test_v2.json")
print("v1 (untightened) reparsed queries -- note the invented comparison axis:")
for r in v1[:2]:
    print(" ", r["reparsed_query"][:200])
print(f"  mean answer length: {sum(len(r['answer']) for r in v1) / len(v1):.0f} chars")
print()
print("v2 (tightened) reparsed queries -- single question, no invented axis:")
for r in v2[:2]:
    print(" ", r["reparsed_query"])
print(f"  mean answer length: {sum(r['answer_len'] for r in v2) / len(v2):.0f} chars")


v1 (untightened) reparsed queries -- note the invented comparison axis:
  1. ما هي جميع الفئات من الورثة المستحقين لراتب أو معاش بموجب قانون الضمان الاجتماعي الأردني في حالة وفاة الموظف أثناء عمله، وما الشروط المحددة لكل فئة على حدة؟

2. هل هناك فرق في الاستحقاق والشروط بين
  1. ما هي جميع فئات الورثة الذين قد يستحقون راتباً أو معاشاً بعد وفاة الزوج أثناء عمله، وما الشروط المحددة لكل فئة على حدة (الزوجة، الأطفال، الآباء، وغيرهم إن وجدوا)؟

2. هل يختلف الاستحقاق بناءً على ن
  mean answer length: 5036 chars

v2 (tightened) reparsed queries -- single question, no invented axis:
  ما هي الشروط التي يجب توفرها جميعها حتى يستحق ورثة الموظف المتوفى أثناء مباشرته للعمل على راتب أو معاش من صندوق التأمين الاجتماعي أو صاحب العمل، وما هي الفئات المختلفة من الورثة التي قد تكون مستحقة لهذا الراتب والشروط الخاصة بكل فئة؟
  ما هي جميع الشروط التي يجب توفرها لكي يستحق ورثة الموظف المتوفى أثناء عمله على راتب أو استحقاقات مالية، مع بيان الشروط المتعلقة بكل فئة من فئات الورثة المختلفة (الزوجة، والأطفال، و

## 2. q29 (heir eligibility conditions, Article 81) -- the target case

Three independent rounds, 18 trials total: the untightened prompt (standalone
script), the tightened prompt (standalone script), and the tightened prompt through
the actual production `generate()` path (append mechanism, as shipped). All three
check for the specific content the original defect omitted: the age-23/disability
exception for sons, the marriage-suspends/divorce-resumes rule, and the husband's
disability-plus-no-income condition.


In [3]:
def score_q29(answer):
    # Arabic digit "23" or its word form ("الثلاث والعشرين") -- one trial in the
    # first pass used the word form and false-negatived on a bare "23" search.
    has_age23 = ("23" in answer) or ("والعشرين" in answer and "ثلاث" in answer)
    has_marriage = ("زواج" in answer) or ("marri" in answer.lower())
    # husband condition: match either "العجز الكلي" or the English phrasing, avoiding
    # the earlier false-negative from checking the substring without the "ال" prefix
    has_husband = (("عجز" in answer and ("أجر" in answer or "دخل" in answer))
                   or ("total disability" in answer.lower() and "income" in answer.lower()))
    return has_age23 and has_marriage and has_husband

prod_path = load("q29_reparser_production_path.json")
rounds = {"v1 (untightened)": v1, "v2 (tightened)": v2, "production path (tightened, append)": prod_path}
for name, rows in rounds.items():
    hits = sum(score_q29(r["answer"]) for r in rows)
    print(f"{name}: {hits}/{len(rows)} complete")


v1 (untightened): 6/6 complete
v2 (tightened): 6/6 complete
production path (tightened, append): 6/6 complete


**18/18 across three independent rounds** (two apparent misses in the raw automated
checks during the live investigation were both false negatives -- one keyword search
that didn't account for Arabic's definite article prefix, one that missed a
word-form number instead of digits -- both confirmed by reading the actual answer
text and fixed in the scorer above). Compare to the prior state: 0/9 with no
intervention across every prior measurement (Phase 31, Phase 34), ~3-4/12 (~30%)
with the broadened `SYSTEM_PROMPT` rule 5 alone (Phase 35). The reparser is a
stronger, more reliable fix for this specific defect than the prompt rule was.

**Cost, not free:** mean answer length roughly 2,000-4,700 characters vs. ~800-1,500
without it -- because a genuinely more explicit question legitimately triggers more
of `SYSTEM_PROMPT`'s coverage rules (rule 4), pulling in adjacent articles (Article
84's pension-combination rules) that are arguably relevant but weren't asked about
directly. Not clearly a bug, not clearly free -- disclosed as a real tradeoff.


## 3. Regression check -- does it inflate answers that were already fine?

Re-ran three previously-clean questions (q01, q35, q42) plus a deliberately simple,
single-fact question, all through the shipped `generate()` path with the reparser
active.


In [4]:
reg = load("reparser_regression_check.json")
baseline_lens = {"q01": 2151, "q35": 1320, "q42": 1424}  # from the pre-reparser Phase 34 recheck
for qid, d in reg.items():
    base = baseline_lens.get(qid)
    delta = f" (baseline {base}, {(d['len']/base - 1)*100:+.0f}%)" if base else ""
    print(f"{qid}: {d['len']} chars{delta}")


q01: 2869 chars (baseline 2151, +33%)
q35: 1452 chars (baseline 1320, +10%)
q42: 1482 chars (baseline 1424, +4%)
q_simple: 1068 chars


Length grew (q01 +33%, q35 +10%, q42 +4%), but reading the actual answers: no
padding or invented tangents. q01's growth is a genuine, arguably useful addition
(it now proactively states the Social-Security-coverage precondition up front,
matching Phase 28's rule 21/22 intent). The simple leave-days question stayed
appropriately short and correctly asked for the one missing fact (years of service)
rather than padding. **No clear regression found**, but the reparser does add cost
and length to questions that did not need it -- it runs unconditionally on every
question, not just multi-category ones, since building a reliable "should I bother"
trigger was out of scope for this pass.


## 4. Could the reparser have replaced the OTHER bespoke fixes already in `calc.py`?

Three targeted tests, each with every deterministic calculator monkeypatched to
return `None` (so nothing but the reparser could help) while the reparser itself
stayed active -- isolating what the reparser alone can and cannot do.


### 4a. Contributions arithmetic (the original Phase 24 problem)

Ground truth, read directly from the articles: on a 1000 JD salary, the employee
share is 7.5% (75 JD: 6.5% old-age + 1% unemployment) and the employer share is
14.25% (142.5 JD). Phase 24's finding, before `compute_contributions` existed: the
model got this right only 3-6 times out of 6 even with every needed article in
context, because it requires combining four articles, attributing each rate to the
correct side, and summing correctly -- several chained steps, not a single fact to
notice.


In [5]:
contrib = load("reparser_vs_contributions.json")
# hand-scored while reading the actual trials during the investigation
verdicts = {
    0: ("WRONG", "old-age rate misstated as 5.5% (actual 6.5%); employee total 65 JD, should be 75"),
    1: ("correct", "65 + 10 = 75 JD, matches ground truth"),
    2: ("correct", "65 + 10 = 75 JD, matches ground truth"),
    3: ("WRONG", "maternity insurance (employer-only) wrongly added to employee side; "
                  "header figure (83.5) doesn't even match its own table (82.5)"),
    4: ("WRONG", "claims old-age insurance (6.5%, 65 JD) is not deducted from the employee at all -- "
                  "states only 10 JD owed, the same understated-total failure Phase 24 was built to fix"),
    5: ("correct", "65 + 10 = 75 employee, 110 + 20 + 7.5 + 5 = 142.5 employer, 217.5 total -- fully correct"),
}
for t, (v, why) in verdicts.items():
    print(f"trial {t}: {v} -- {why}")
n_correct = sum(1 for v, _ in verdicts.values() if v == "correct")
print(f"\n{n_correct}/6 correct -- matches Phase 24's original 3-6/6 pre-calculator baseline almost exactly.")


trial 0: WRONG -- old-age rate misstated as 5.5% (actual 6.5%); employee total 65 JD, should be 75
trial 1: correct -- 65 + 10 = 75 JD, matches ground truth
trial 2: correct -- 65 + 10 = 75 JD, matches ground truth
trial 3: WRONG -- maternity insurance (employer-only) wrongly added to employee side; header figure (83.5) doesn't even match its own table (82.5)
trial 4: WRONG -- claims old-age insurance (6.5%, 65 JD) is not deducted from the employee at all -- states only 10 JD owed, the same understated-total failure Phase 24 was built to fix
trial 5: correct -- 65 + 10 = 75 employee, 110 + 20 + 7.5 + 5 = 142.5 employer, 217.5 total -- fully correct

3/6 correct -- matches Phase 24's original 3-6/6 pre-calculator baseline almost exactly.


**The reparser does NOT fix this.** 3/6 correct, statistically indistinguishable
from the exact pre-`calc.py` baseline this project measured in Phase 24. Confirms
the theoretical distinction directly: a reparse step that makes a question's SCOPE
more explicit does nothing for a failure that isn't about scope -- summing four
rates correctly and attributing each to the right side is an arithmetic-reliability
problem, and no amount of "did you consider every case" framing gives the model a
calculator. `compute_contributions` is not replaceable by this mechanism.


### 4b. Sex-differentiated pension split (mt02 -- the other calc.py fix)

The exact mt02 scenario: told "Male" as a one-word reply after being asked, must
answer using only the male figure (252 contributions) without leaking the female
figure (228) or re-asking.


In [6]:
gender = load("reparser_vs_gender.json")
def score_gender(answer):
    # NOT "228 not in answer" -- 228 is also the correct MALE figure for a
    # different sub-clause (Article 64(b)(1)'s transitional route), so its mere
    # presence is not a leak. The real signal: the word "female" never appears
    # at all when the answer stays correctly male-only.
    never_states_female = "female" not in answer.lower()
    acknowledges_male = "male" in answer.lower()
    return never_states_female and acknowledges_male

hits = sum(score_gender(r["answer"]) for r in gender)
print(f"{hits}/{len(gender)} correctly narrowed to the male-only figure, female figure never leaked in")


6/6 correctly narrowed to the male-only figure, female figure never leaked in


**6/6 -- the reparser DOES fix this, matching `compute_gender_conditions`'s own
reliability.** Unlike the arithmetic case, this failure genuinely is a "notice and
apply an already-stated fact" problem -- the same class of failure the reparser
fixed for q29 -- so it generalises here too. Interesting result: two structurally
different fixes (a deterministic code extractor vs. a general reparse step) both
reach full reliability on the same underlying defect class.


### 4c. q09 (Article 40 vs Article 85) -- with the disambiguation note also disabled

Same question, same retrieved context (both Article 40 and Article 85 present),
reparser active, `disability_pension_scope_note` disabled.


In [7]:
q09 = load("reparser_vs_q09.json")
def score_q09(answer):
    # correct: leads with / relies on Article 40 without also presenting a
    # contradictory "cannot combine" framed around Article 85 as if it were the
    # governing rule for a WORK-INJURY pension
    mislabels = "work-related disability" in answer.lower() and "85(e)" in answer
    return not mislabels

hits = sum(score_q09(r["answer"]) for r in q09)
print(f"{hits}/{len(q09)} avoided the Article 85 mislabeling")
for r in q09:
    mislabels = "work-related disability" in r["answer"].lower() and "85(e)" in r["answer"]
    print(f"  trial {r['trial']}: {'MISLABELS 85(e) as work-related' if mislabels else 'clean'}")


0/6 avoided the Article 85 mislabeling
  trial 0: MISLABELS 85(e) as work-related
  trial 1: MISLABELS 85(e) as work-related
  trial 2: MISLABELS 85(e) as work-related
  trial 3: MISLABELS 85(e) as work-related
  trial 4: MISLABELS 85(e) as work-related
  trial 5: MISLABELS 85(e) as work-related


**0/6 -- the reparser does NOT fix this, and it is not a completeness problem.**
Every trial still presents Article 85(e)/(f) -- which the retrieved text itself
labels "Natural Disability Pension" -- as though it governs a "Total/Partial
Permanent WORK-RELATED Disability Pension," the identical mislabeling measured
before any fix existed. Asking the model to "consider every relevant case" does not
help when the actual failure is picking the WRONG case as relevant in the first
place -- Article 85 keeps getting cited as if it were about work-injury disability
because it sits right next to Article 40 in retrieval and shares surface vocabulary
("disability pension," "combine... with wages"), not because the model failed to
notice enough content. `disability_pension_scope_note` is not replaceable by this
mechanism either.


## Verdict -- a three-way taxonomy, not a universal fix

| Failure class | Example | Reparser alone | What actually fixes it |
|---|---|---|---|
| **Completeness** -- a dense article has several nested cases and the model summarises instead of enumerating | q29 (Article 81 heir conditions) | **Fixed, 18/18** | Reparser (adopted) |
| **Precision/narrowing** -- a fact is already known but the model states more than that one case | mt02 (sex-differentiated pension split) | **Fixed, 6/6** | Either the reparser OR the existing `compute_gender_conditions` extractor -- both reach full reliability |
| **Arithmetic** -- several rates must be attributed to the right side and summed correctly | contributions (Phase 24) | **NOT fixed, 3/6** -- matches the pre-calculator baseline exactly | Only `compute_contributions` (deterministic extraction) |
| **Mislabeling** -- two similarly-worded, adjacent provisions govern different things and the wrong one gets cited | q09 (Article 40 vs 85) | **NOT fixed, 0/6** -- identical failure to no-fix-at-all | Only `disability_pension_scope_note` (targeted disambiguation) |

**The reparser is adopted into production regardless** (per direct instruction),
wired unconditionally into `generate()` -- it is a genuine, well-verified improvement
for completeness-class failures generally, not just q29, and may catch similar gaps
in articles this project hasn't specifically found and fixed yet. But it does not
generalise to every failure this project has needed a custom fix for, and the two
classes it fails on (arithmetic, mislabeling) are exactly the two classes
`calc.py`'s existing fixes were built for -- **none of the existing bespoke fixes
are being removed.** The real, disclosed costs: roughly doubled LLM calls per
question, answers ~1.1x-3x longer even on already-good questions, and it currently
runs on every question rather than only where it is likely to help.
